# 导入依赖和定义文件

In [1]:
import time
import json
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import joblib
import optuna

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)

# 寻找并保存最优参数

In [3]:
# 定义 LGBM 的目标函数
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1,log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'num_leaves': trial.suggest_int('num_leaves', 20, 30),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)

# 定义 CatBoost 的目标函数
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1,log=True),
        'depth': trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 3),
        'border_count': trial.suggest_categorical('border_count', [32, 64]),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)

# 创建 Optuna 研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

# 获取最优参数
lgbm_best_params = lgbm_study.best_params
catboost_best_params = catboost_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_value = catboost_study.best_value

[I 2025-04-26 17:56:53,800] A new study created in memory with name: no-name-8eebe131-491e-4fa8-89c5-3ab86dd1d1a5
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:53,845] Trial 0 finished with value: 0.7924094307073031 and parameters: {'n_estimators': 50, 'learning_rate': 0.0704343411216729, 'max_depth': 6, 'num_leaves': 26, 'subsample': 0.9958342524585022, 'colsample_bytree': 0.8268435499204921}. Best is trial 0 with value: 0.7924094307073031.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:53,906] Trial 1 finished with value: 0.7912593444508338 and parameters: {'n_estimators': 100, 'l

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000501 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000506 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] N

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:54,057] Trial 4 finished with value: 0.78953421506613 and parameters: {'n_estimators': 100, 'learning_rate': 0.03416633295534978, 'max_depth': 6, 'num_leaves': 28, 'subsample': 0.9824945774458721, 'colsample_bytree': 0.920005487900532}. Best is trial 2 with value: 0.79700977573318.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:54,099] Trial 5 finished with value: 0.7958596894767107 and parameters: {'n_estimators': 100, 'learning_rate': 0.05709001003001779, 'max_depth': 4, 'num_leaves': 25, 'subsample': 0.8231232886648597, 'colsample_byt

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000427 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:54,268] Trial 8 finished with value: 0.7688326624496837 and parameters: {'n_estimators': 100, 'learning_rate': 0.0100538373534507, 'max_depth': 5, 'num_leaves': 27, 'subsample': 0.9355852086101366, 'colsample_bytree': 0.8901028079158311}. Best is trial 2 with value: 0.79700977573318.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:54,324] Trial 9 finished with value: 0.7924094307073031 and parameters: {'n_estimators': 100, 'learning_rate': 0.03965942661933663, 'max_depth': 5, 'num_leaves': 24, 'subsample': 0.939256769593909, 'colsample_by

...

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:56,399] Trial 45 finished with value: 0.7929844738355377 and parameters: {'n_estimators': 100, 'learning_rate': 0.06024242277622015, 'max_depth': 5, 'num_leaves': 23, 'subsample': 0.8069400536979177, 'colsample_bytree': 0.932049826174947}. Best is trial 31 with value: 0.80448533640023.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:56,460] Trial 46 finished with value: 0.7993099482461185 and parameters: {'n_estimators': 100, 'learning_rate': 0.07353404487226818, 'max_depth': 5, 'num_leaves': 21, 'subsample': 0.8209381243911151, 'colsampl

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 17:56:56,631] Trial 49 finished with value: 0.8050603795284647 and parameters: {'n_estimators': 100, 'learning_rate': 0.09023376732808969, 'max_depth': 6, 'num_leaves': 21, 'subsample': 0.8524080369512816, 'colsample_bytree': 0.9678797899503775}. Best is trial 49 with value: 0.8050603795284647.
[I 2025-04-26 17:56:56,632] A new study created in memory with name: no-name-091d6250-37ae-4ff0-ae3c-a2b46b02b73a


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000306 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

[I 2025-04-26 17:56:56,881] Trial 0 finished with value: 0.765382403680276 and parameters: {'n_estimators': 100, 'learning_rate': 0.010945752179540193, 'depth': 4, 'l2_leaf_reg': 2, 'border_count': 64}. Best is trial 0 with value: 0.765382403680276.
[I 2025-04-26 17:56:57,086] Trial 1 finished with value: 0.7952846463484762 and parameters: {'n_estimators': 100, 'learning_rate': 0.029786199308235242, 'depth': 5, 'l2_leaf_reg': 1, 'border_count': 32}. Best is trial 1 with value: 0.7952846463484762.
[I 2025-04-26 17:56:57,306] Trial 2 finished with value: 0.7924094307073031 and parameters: {'n_estimators': 100, 'learning_rate': 0.02818751488236258, 'depth': 5, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 1 with value: 0.7952846463484762.
[I 2025-04-26 17:56:57,468] Trial 3 finished with value: 0.7935595169637722 and parameters: {'n_estimators': 50, 'learning_rate': 0.09201363802004797, 'depth': 6, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 1 with value: 0.795284646348476

In [4]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)

# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.8050603795284647
CatBoost 最佳得分: 0.8016101207590569


# 训练并保存模型

In [6]:
# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)


lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)


joblib.dump(LGBMClassifier, 'lgbm_best_model.joblib')
joblib.dump(CatBoostClassifier, 'catboost_best_model.joblib')

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000654 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

['catboost_best_model.joblib']

# 提交文件

In [7]:
# 加载模型
lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
catboost_loaded_model = joblib.load('catboost_best_model.joblib')


# 对测试数据进行预测，获取概率值
lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
catboost_prob = catboost_model.predict_proba(X_test)[:, 1]

# 简单平均融合
ensemble_prob = (lgbm_prob + catboost_prob) / 2

# 根据阈值生成最终预测
threshold = 0.5
ensemble_pred = ensemble_prob > threshold

# 加载测试数据
test_data = pd.read_csv('test.csv')  # 确保文件路径正确

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
    'Transported': ensemble_pred
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
